# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [ ]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/Andersen_NCBI_Virus_GISAID/"

os.chdir(downloads)

## Collect user input

In [ ]:
locations = input("Locations (separate with commas and no spaces): ")
start_date = input("Start date (format: YYYY-MM-DD): ")
end_date = input("End date (format: YYYY-MM-DD): ")

## Create directories if needed

In [ ]:
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + "_".join(locations.split(",")) + "/"
andersen_ncbi_virus = home + "Combinations/Andersen_NCBI_Virus/" + start_date + "--" + end_date + "_" + "_".join(locations.split(",")) + "/"
complete_files = andersen_ncbi_virus_gisaid + start_date + "--" + end_date + "_" + "_".join(locations.split(",")) + "/"

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

## Get list of genotypes and states

In [5]:
# Get list of genotypes and states

# os.chdir("C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/")
os.chdir(references)

states = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["D1.1"] # ["B3.13", "D1.1", "D1.3"] # , "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

## Download all files, run through all files, convert fasta files to dataframes, and separate them into different dataframes based on segment ##

In [ ]:
all_metadata_files = []
all_fasta_files = []

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    else: # If we don't have any downloaded files
        # Have user type in username and password
        username = input("Username: ")
        password = input("Password: ")
        browser = input("Browser: ")
        sleep_time = input("Seconds to sleep in between clicks: ")

        open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 


for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name, engine="xlrd")
            all_metadata_files.append(metadata)

        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            # print(fasta_file[fasta_file["Geo_Location"] != "USA"])
            # break 
            all_fasta_files.append(fasta_file)
    break 

print(len(all_metadata_files))
print(len(all_fasta_files))

# print(all_metadata_files)

# all_metadata_files = [all_metadata_files[0]]
# all_fasta_files = [all_fasta_files[1]]

16
16


In [7]:
# Get metadata

def separate_fasta_by_segs(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first
    # Dummy host type -- we'll actually add this in later
    # b313_fasta["Host_Type"] = "other"
    # d11_fasta["Host_Type"] = "other"

    unique_segments = list(set(fasta["Segment"])) # Get list of segments
    # genotypes = ["B3.13", "D1.1"]
    # genotype_fastas = {"B3.13": b313_fasta, "D1.1": d11_fasta}

    # “>EPI_ID/Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for fasta_gen in genotypes: # .keys(): # For each genotype
        # print(fasta_gen)
        for seg in unique_segments: # For each segment

            xls = metadata[metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == fasta_gen] # Get only the metadata corresponding to that genotype

            # print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Isolate_Id"])

            # print(d11_xls)

            # FASTA
            # if "Identifier" in fasta.columns:
            #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
            # else:           
            #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])

            fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # [mask] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            # print(fasta_seg_pre)

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = fasta_gen

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name"] + "|" + fasta_seg["Subtype"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] + "\n"
            fasta_seg["New_Name"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            # print(fasta_seg)

    return segment_fastas, unique_segments


# Separate fastas by segment
segment_fastas = []
# unique_segments = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):
    # print(all_fasta_files)
    # print(fasta)
    metadata = all_metadata_files[i]
    # print(metadata)
    # print(fasta.loc[i, "Isolate_Name"])
    
    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segs(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment
    # print(fastas[0])
    segment_fastas.append(fastas)

# print(segment_fastas[0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_32764\3365780067.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_32764\3365780067.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_32764\3365780067.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

## De-Duplication

In [ ]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {}
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
        fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
        andersen_ncbi[segment_genotype] = fasta_file
    

In [10]:
# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

def partial_isolate(id):

    partial = id.split("_")[-1] # If 25_, get the last bit
    digits = partial.split("-")
    # Build partial isolates
    isolate = ""
    other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        elif d.isnumeric() == False: # If it's a weird isolate
            other = d + "-"
        else: 
            other = other + d
    # Now add to list to check in Andersen files without doing wild for loops
    if len(isolate) == 10: # If this is a correctly formatted isolate
        # isolates.append(isolate)
        # All headers are followed by sequences
        partial_isolate = isolate
    else: # If this is some other isolate
        partial_isolate = other

    return partial_isolate

ha_only = [] # Only do one segment, as the others are identical 
for gisaid_fasta in segment_fastas:
    for genotype_gisaid_fasta in gisaid_fasta:
        # print(genotype_gisaid_fasta)
        if "HA" in genotype_gisaid_fasta["Segment"].values:
            genotype_gisaid_fasta["Partials"] = genotype_gisaid_fasta["Isolate_Id"].apply(partial_isolate)
            genotype_gisaid_fasta["Year"] = genotype_gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year))
            genotype_gisaid_fasta = genotype_gisaid_fasta.drop_duplicates(keep="first")
            ha_only.append(genotype_gisaid_fasta)
        # print(genotype_gisaid_fasta)
        
# print(ha_only)
    
andersen_ncbi_genotypes = {}
for key in andersen_ncbi:
    andersen_ncbi_fasta = andersen_ncbi[key]
    # Find partial Isolate IDs
    andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    print("Andersen:", andersen_ncbi_fasta["Partials"])
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year))
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(keep="first")
    if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys() and andersen_ncbi_fasta["Segment"].values[0] == "HA": # If we haven't already seen this genotype, and if HA
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype dictionary
        # print(andersen_ncbi_fasta)
    # break 

# print(andersen_ncbi_genotypes)

gisaid_only_dfs = {} # We need these so that we know what sequences to throw out in the other segments too
for gisaid_genotype in ha_only: # Each dataframe is unique in genotype
    print(gisaid_genotype[gisaid_genotype["Host_Type"] == "human"])
    genotype = gisaid_genotype["Genotype"].values[0]
    andersen_ncbi_fasta = pd.DataFrame()
    if genotype in andersen_ncbi_genotypes.keys() and genotype not in gisaid_only_dfs.keys(): # If it's in Andersen and we haven't seen it before here
        andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
        # print(andersen_ncbi_fasta)
        deduplicated = pd.concat([gisaid_genotype,andersen_ncbi_fasta]).drop_duplicates(subset=["Partials", "Year"], keep="first") 
        
        gisaid_only_dfs[genotype] = deduplicated
    # else:
    #     deduplicated = gisaid_genotype
    
# print(gisaid_only_dfs["D1.3"][gisaid_only_dfs["D1.3"]["Host_Type"] == "cattle"])
    
        print(deduplicated[deduplicated["Host_Type"] == "human"])

# Identify sequences we are keeping
genotype_seq_keep = {}
for genotype in gisaid_only_dfs:
    deduplicated = gisaid_only_dfs[genotype]
    genotype_seq_keep[genotype] = list(deduplicated["Identifier"])

# Keep in other segments only the sequences we kept in HA
kept_seqs = []
for genotype_group in segment_fastas:
    # print(genotype_group)
    for gisaid_df in genotype_group:
        # print(gisaid_df)
        if len(gisaid_df["Genotype"]) > 0: # If there are sequences
            genotype = gisaid_df["Genotype"].values[0]
            gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]
            print(gisaid_df_new)
            kept_seqs.append(gisaid_df_new)

print(kept_seqs)

Andersen: 0       032809-002
1       032809-001
2       032297-002
3       032297-001
4       032296-002
           ...    
2865    019408-003
2866    019408-002
2867    019408-001
2868    013149-001
2869    013300-001
Name: Partials, Length: 2870, dtype: object
Andersen: 0       032809-002
1       032809-001
2       032297-002
3       032297-001
4       032296-002
           ...    
2865    019408-003
2866    019408-002
2867    019408-001
2868    013149-001
2869    013300-001
Name: Partials, Length: 2870, dtype: object
Andersen: 0       032809-002
1       032809-001
2       032297-002
3       032297-001
4       032296-002
           ...    
2865    019408-003
2866    019408-002
2867    019408-001
2868    013149-001
2869    013300-001
Name: Partials, Length: 2870, dtype: object
Andersen: 0       032809-002
1       032809-001
2       032297-002
3       032297-001
4       032296-002
           ...    
2865    019408-003
2866    019408-002
2867    019408-001
2868    013149-001
2869    013

Create animal reference if needed 

In [11]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(home)
animals_df.to_csv("animals_ref_to_sort.csv")

['pelecanus', 'black_crowned_night-heron', 'antarctic_tern', 'great_grey_owl', 'british_columbia', 'red_necked_phalarope', 'mute_swan', 'red-tailed_hawk', 'western_sandpiper', 'wild_duck', "swanson's_hawk", 'osprey', 'cape_petrel', 'megascops_choliba', 'king_vulture', 'blue_jay', 'chile', 'grackle', 'humboldt_penguin', 'ring-tailed_hawk', 'laughing_gull', 'south_american_sea_lion', 'colorado', 'swallow', 'michigan', 'pheasant', 'gadwall', 'short_billed_gull', 'domestic_duck', 'burmeisters_porpoise', 'gull', 'mountain_lion', 'backyard_chicken', 'great_grebe', 'golden_eagle', 'embden_goose', 'snowy_egret', 'environment', 'feral_cat', 'peruvian_booby', 'franklins_gull', 'lesser_scaup', 'poultry', 'domestic_turkey', 'northwestern_crow', 'double-crested_cormorant', 'american_kestrel', 'chukar', 'american_crow', 'lynx', 'ring-necked_pheasant', 'magpie', 'scoter', 'parrot', 'greater_scaup', 'gyrfalcon', 'kelp_gull', 'mixed', "lady_amherst's_pheasant", 'layer_chicken', 'crow', 'wild_mink', 'bl

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [ ]:
# 16 files needed
huge_fasta = pd.DataFrame()

for fastas in kept_seqs: # 7 batches
    # print(fastas.columns)
    # print(len(fastas))
    # break
    # for f in fastas: # 16 files per batch 
        # print(f)
        # break 
    huge_fasta = pd.concat([huge_fasta, fastas])

print(huge_fasta.columns)

# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

# genotypes = ["B3.13", "D1.1"] # , "D1.3"]
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)

print(big_fasta)

# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    fasta_df = fasta[["New_Name", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = complete_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_GISAID_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            output_file.write(item)
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

print(len(huge_fasta))
print(len(big_fastas[0]))

Index(['Header', 'Isolate_Id', 'Isolate_Name', 'Subtype', 'Segment',
       'Geo_Location', 'Date Collected', 'Species', 'Sequence', 'Identifier',
       'Host_Type', 'Genotype', 'New_Name', 'Partials', 'Year'],
      dtype='object')
                                                  Header        Isolate_Id  \
24     EPI_ISL_19743149|A/vulture/Maryland/001987-002...        001987-002   
40     EPI_ISL_19743148|A/vulture/Maryland/001987-003...        001987-003   
72     EPI_ISL_19743150|A/vulture/Maryland/001987-001...        001987-001   
104    EPI_ISL_19743144|A/barn_owl/California/003273-...        003273-002   
120    EPI_ISL_19743147|A/vulture/California/003065-0...        003065-001   
...                                                  ...               ...   
11790  EPI_ISL_19744977|A/turkey/Idaho/000070-001/202...        000070-001   
11798  EPI_ISL_19744976|A/turkey/Idaho/000070-002/202...        000070-002   
11806  EPI_ISL_19744979|A/chicken/Idaho/24-036514-001...  24-036

## Concatenate to Andersen_NCBI files and save

In [ ]:
# Concat
os.chdir(andersen_ncbi_virus_gisaid)

for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        for dirpath1, dirs1, files1 in os.walk(complete_files):
            for file1 in files1:
                file_name1 = os.path.join(dirpath1, file1)
                # print(file_name1)
                
                if "_".join(file_name.split("/")[-1].split("_")[0:2]) == "_".join(file_name1.split("/")[-1].split("_")[0:2]): # If they match
                    print("_".join(file_name.split("/")[-1].split("_")[0:2]))
                    output_path = andersen_ncbi_virus_gisaid + "all_" + file_name.split("/")[-1] # Genotype and Segment should all be the same
                    output_file = open(output_path, "w")
                    with open(file_name) as f:
                        for line in f.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                        f.close()
                    with open(file_name1) as f1:
                       for line in f1.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                    f1.close()  

                    output_file.close()
                

            break 
    break 

D1.1_HA
D1.1_HA
D1.1_HA
D1.1_MP
D1.1_MP
D1.1_MP
D1.1_NA
D1.1_NA
D1.1_NA
D1.1_NP
D1.1_NP
D1.1_NP
D1.1_NS
D1.1_NS
D1.1_NS
D1.1_PA
D1.1_PA
D1.1_PA
D1.1_PB1
D1.1_PB1
D1.1_PB1
D1.1_PB2
D1.1_PB2
D1.1_PB2
